In [1]:
# ── Imports, chargement et merge ─────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# 2021/2022/2023 → séparateur point-virgule (;)
# 2024 → séparateur virgule (,)
# Num_Acc forcé en string pour éviter les problèmes de jointure
# Colonne annee ajoutée pour filtrer par année dans les analyses

df_2021 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - vehicules 2021.csv', sep=';', dtype={'Num_Acc': str})
df_2022 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - vehicules 2022.csv', sep=';', dtype={'Num_Acc': str})
df_2023 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - vehicules 2023.csv', sep=';', dtype={'Num_Acc': str})
df_2024 = pd.read_csv('Dataset/Dataset - Sécurité Routière.xlsx - vehicules 2024.csv', sep=',',  dtype={'Num_Acc': str})

df_2021['annee'] = 2021
df_2022['annee'] = 2022
df_2023['annee'] = 2023
df_2024['annee'] = 2024

vehicules_df = pd.concat([df_2024, df_2023, df_2022, df_2021], ignore_index=True)
print(f"✅ Merge OK — {vehicules_df.shape[0]:,} lignes x {vehicules_df.shape[1]} colonnes")
vehicules_df.head()

✅ Merge OK — 378,071 lignes x 12 colonnes


,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc,annee
0,202400000001,155 781 758,A01,1,7,0,2,1,13,1,NaN,2024
1,202400000001,155 781 759,B01,2,14,0,2,2,21,1,NaN,2024
2,202400000002,155 781 757,A01,1,10,0,1,3,15,1,NaN,2024
3,202400000003,155 781 756,A01,3,3,9,1,1,1,1,NaN,2024
4,202400000004,155 781 754,B01,3,34,0,2,0,0,1,NaN,2024


In [2]:
# ── Suppression des colonnes inutiles ───────────────────────────
# senc   → sens de circulation, redondant avec les données de lieux
# choc   → point de choc initial, trop granulaire pour nos analyses
# manv   → manœuvre avant accident, trop granulaire
# occutc → nombre d'occupants transport en commun, 99% NaN

cols_to_drop = ['senc', 'choc', 'manv', 'occutc']
vehicules_df = vehicules_df.drop(columns=cols_to_drop)
print(f"✅ Colonnes supprimées — shape : {vehicules_df.shape}")
print(f"Colonnes restantes : {list(vehicules_df.columns)}")

✅ Colonnes supprimées — shape : (378071, 8)
Colonnes restantes : ['Num_Acc', 'id_vehicule', 'num_veh', 'catv', 'obs', 'obsm', 'motor', 'annee']


In [3]:
# ── Remplacement des -1 par NaN ─────────────────────────────────
# Dans le fichier BAAC, -1 signifie "non renseigné"
# On remplace à la fois les -1 numériques ET les "-1" string

cols_sentinel = ['catv', 'obs', 'obsm', 'motor']

for col in cols_sentinel:
    vehicules_df[col] = vehicules_df[col].replace(-1, np.nan)
    vehicules_df[col] = vehicules_df[col].replace("-1", np.nan)

print("✅ Valeurs -1 remplacées par NaN")
print(vehicules_df[cols_sentinel].isna().sum())

✅ Valeurs -1 remplacées par NaN
catv      12
obs      176
obsm     162
motor    711
dtype: int64


In [4]:
# ── Renommage des colonnes pour plus de lisibilité ───────────────
# On renomme pour que les colonnes soient compréhensibles
# sans avoir besoin du dictionnaire BAAC

vehicules_df = vehicules_df.rename(columns={
    'catv' : 'categorie_vehicule',
    'obs'  : 'obstacle_fixe',
    'obsm' : 'obstacle_mobile',
    'motor': 'motorisation'
})

print(f"✅ Colonnes renommées")
print(f"Colonnes : {list(vehicules_df.columns)}")

✅ Colonnes renommées
Colonnes : ['Num_Acc', 'id_vehicule', 'num_veh', 'categorie_vehicule', 'obstacle_fixe', 'obstacle_mobile', 'motorisation', 'annee']


In [5]:
# ── Vérification finale avant export ────────────────────────────

print(f"Shape final : {vehicules_df.shape}")

print("\n--- Types des colonnes ---")
print(vehicules_df.dtypes)

print("\n--- NaN par colonne ---")
nan_df = pd.DataFrame({
    'NaN count': vehicules_df.isna().sum(),
    'NaN %': (vehicules_df.isna().mean() * 100).round(1)
})
print(nan_df[nan_df['NaN count'] > 0])

print("\n--- Vérif aucun -1 restant ---")
for col in vehicules_df.select_dtypes(include='number').columns:
    n = (vehicules_df[col] == -1).sum()
    if n > 0:
        print(f"  ⚠️  {col} : {n} valeurs -1 restantes")
print("✅ Aucun -1 restant")

print("\n--- Doublons ---")
print(f"Doublons Num_Acc : {vehicules_df.duplicated('Num_Acc').sum()}")

Shape final : (378071, 8)

--- Types des colonnes ---
Num_Acc                object
id_vehicule            object
num_veh                object
categorie_vehicule    float64
obstacle_fixe         float64
obstacle_mobile       float64
motorisation          float64
annee                   int64
dtype: object

--- NaN par colonne ---
                    NaN count  NaN %
categorie_vehicule         12    0.0
obstacle_fixe             176    0.0
obstacle_mobile           162    0.0
motorisation              711    0.2

--- Vérif aucun -1 restant ---
✅ Aucun -1 restant

--- Doublons ---
Doublons Num_Acc : 157027


In [6]:
# ── Export du fichier nettoyé ───────────────────────────────────
# Export avec séparateur point-virgule (;) cohérent avec les autres tables
# Ce fichier sera utilisé pour la jointure avec caract et usagers sur Num_Acc

vehicules_df.to_csv('Dataset/vehicules_2021_2024_clean.csv', index=False, sep=';', encoding='utf-8')
print(f"✅ Exporté — {len(vehicules_df):,} lignes x {vehicules_df.shape[1]} colonnes")

✅ Exporté — 378,071 lignes x 8 colonnes


In [7]:
# Vérifier si les doublons ont des valeurs différentes
doublons = vehicules_df[vehicules_df.duplicated('Num_Acc', keep=False)]
print(f"Lignes concernées : {len(doublons)}")
print(doublons.head(6))

Lignes concernées : 294384
        Num_Acc  id_vehicule num_veh  categorie_vehicule  obstacle_fixe  \
0  202400000001  155 781 758     A01                 7.0            0.0   
1  202400000001  155 781 759     B01                14.0            0.0   
4  202400000004  155 781 754     B01                34.0            0.0   
5  202400000004  155 781 755     A01                 7.0            0.0   
6  202400000005  155 781 750     A01                 7.0            0.0   
7  202400000005  155 781 751     B01                 7.0            0.0   

   obstacle_mobile  motorisation  annee  
0              2.0           1.0   2024  
1              2.0           1.0   2024  
4              2.0           1.0   2024  
5              2.0           0.0   2024  
6              2.0           1.0   2024  
7              2.0           1.0   2024  
